# 🏢 Amazon ML Challenge 2026 — Business Entity Resolution

**Objective:** Given business records from 3 independent data sources with noisy, inconsistent fields, determine which records across sources refer to the same real-world business entity.

**Approach Overview:**
1. **Data Loading & EDA** — Understand the data, noise patterns, and scale
2. **Text Preprocessing** — Normalize business names and addresses
3. **Blocking / Candidate Generation** — TF-IDF + cosine similarity to efficiently find plausible matches (country-partitioned)
4. **Feature Engineering** — Multiple string similarity features for candidate pairs
5. **Matching Model** — Threshold-based classifier tuned for F₀.₅ (precision-heavy)
6. **Validation** — Hold-out evaluation with macro-averaged F₀.₅
7. **Test Inference & Submission** — Generate `matching_results.tsv` and `candidate_pairs.tsv`

**Metric:** F₀.₅ (macro-averaged per S1 entity) — precision is weighted 2× over recall.

In [ ]:
import pandas as pd
import numpy as np
import re
import os
import warnings
from collections import defaultdict
from tqdm.auto import tqdm

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import vstack as sparse_vstack

from rapidfuzz import fuzz
from rapidfuzz.distance import Levenshtein

warnings.filterwarnings('ignore')
tqdm.pandas()

# Paths
BASE_DIR = "student_resource"
TRAIN_DIR = os.path.join(BASE_DIR, "dataset", "train")
TEST_DIR  = os.path.join(BASE_DIR, "dataset", "test")
OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Reproducibility
SEED = 42
np.random.seed(SEED)

print("Setup complete ✓")

## 1. Data Loading

All files are **tab-separated** (`.tsv`). We must use `sep='\t'` to read them correctly.

In [ ]:
# --- Training Data ---
train_s1 = pd.read_csv(os.path.join(TRAIN_DIR, "train_source1.tsv"), sep="\t")
train_s2 = pd.read_csv(os.path.join(TRAIN_DIR, "train_source2.tsv"), sep="\t")
train_s3 = pd.read_csv(os.path.join(TRAIN_DIR, "train_source3.tsv"), sep="\t")
train_gt = pd.read_csv(os.path.join(TRAIN_DIR, "train_ground_truth.tsv"), sep="\t")

# --- Test Data ---
test_s1 = pd.read_csv(os.path.join(TEST_DIR, "test_source1.tsv"), sep="\t")
test_s2 = pd.read_csv(os.path.join(TEST_DIR, "test_source2.tsv"), sep="\t")
test_s3 = pd.read_csv(os.path.join(TEST_DIR, "test_source3.tsv"), sep="\t")

print("=== Training Data ===")
print(f"Source 1: {len(train_s1):,} records")
print(f"Source 2: {len(train_s2):,} records")
print(f"Source 3: {len(train_s3):,} records")
print(f"Ground Truth: {len(train_gt):,} entries")
print()
print("=== Test Data ===")
print(f"Source 1: {len(test_s1):,} records")
print(f"Source 2: {len(test_s2):,} records")
print(f"Source 3: {len(test_s3):,} records")

## 2. Exploratory Data Analysis

In [ ]:
# Check columns
print("S1 columns:", train_s1.columns.tolist())
print("S1 sample:")
display(train_s1.head(5))
print()
print("S2 sample:")
display(train_s2.head(5))
print()
print("S3 sample:")
display(train_s3.head(5))

In [ ]:
# Missing values
print("=== Missing Values (Training) ===")
for name, df in [("S1", train_s1), ("S2", train_s2), ("S3", train_s3)]:
    print(f"\n{name}:")
    print(df.isnull().sum())
    print(f"  Total rows: {len(df):,}")

In [ ]:
# Country distribution
print("=== Country Distribution ===")
for name, df in [("Train S1", train_s1), ("Train S2", train_s2), ("Train S3", train_s3),
                  ("Test S1", test_s1), ("Test S2", test_s2), ("Test S3", test_s3)]:
    print(f"\n{name}:")
    print(df['country'].value_counts())

In [ ]:
# Ground truth analysis
gt = train_gt.copy()
singletons = gt['matched_entity_ids'].isna().sum()
with_matches = len(gt) - singletons
print(f"Total S1 entities: {len(gt):,}")
print(f"Singletons (no match): {singletons:,} ({100*singletons/len(gt):.1f}%)")
print(f"With matches: {with_matches:,} ({100*with_matches/len(gt):.1f}%)")

matched = gt.dropna(subset=['matched_entity_ids'])
match_counts = matched['matched_entity_ids'].str.split(',').apply(len)
print(f"\nMatch count distribution (non-singleton):")
print(f"  Mean: {match_counts.mean():.2f}")
print(f"  Median: {match_counts.median():.0f}")
print(f"  Max: {match_counts.max()}")
print(f"  Distribution:\n{match_counts.value_counts().sort_index()}")

## 3. Text Preprocessing

We normalize business names and addresses to handle common noise patterns:
- Lowercase, strip whitespace
- Standardize legal suffixes (Corp→corporation, Pvt→private, Ltd→limited, etc.)
- Normalize address abbreviations (Rd→road, St→street, etc.)
- Remove extra punctuation and symbols
- Create a combined text field for TF-IDF blocking

In [ ]:
def normalize_name(name):
    """Normalize business name for matching."""
    if pd.isna(name):
        return ""
    s = str(name).lower().strip()
    
    # Remove common noise characters
    s = re.sub(r'[<>\-]+', ' ', s)
    
    # Standardize legal suffixes
    replacements = {
        r'\bpvt\b': 'private',
        r'\bltd\b': 'limited',
        r'\bcorp\b': 'corporation',
        r'\binc\b': 'incorporated',
        r'\bllc\b': 'limited liability company',
        r'\bllp\b': 'limited liability partnership',
        r'\bco\b': 'company',
        r'\bsarl\b': 'sarl',
        r'\bsas\b': 'sas',
        r'\bs\.?a\.?s\b': 'sas',
        r'\bs\.?a\.?r\.?l\b': 'sarl',
        r'\bsci\b': 'sci',
        r'\b&\b': 'and',
        r'\bdba\b': '',
    }
    for pattern, replacement in replacements.items():
        s = re.sub(pattern, replacement, s)
    
    # Remove punctuation except alphanumeric and spaces
    s = re.sub(r'[^\w\s]', ' ', s)
    # Collapse whitespace
    s = re.sub(r'\s+', ' ', s).strip()
    return s


def normalize_address(addr):
    """Normalize business address for matching."""
    if pd.isna(addr):
        return ""
    s = str(addr).lower().strip()
    
    # Standardize common abbreviations
    replacements = {
        r'\brd\b': 'road',
        r'\bst\b': 'street',
        r'\bave\b': 'avenue',
        r'\bblvd\b': 'boulevard',
        r'\bdr\b': 'drive',
        r'\bln\b': 'lane',
        r'\bct\b': 'court',
        r'\bpl\b': 'place',
        r'\bcir\b': 'circle',
        r'\bhwy\b': 'highway',
        r'\bpkwy\b': 'parkway',
        r'\bapt\b': 'apartment',
        r'\bste\b': 'suite',
        r'\bfl\b': 'floor',
        r'\bno\b': 'number',
        r'\bkh\b': 'khasra',
        r'\bdist\b': 'district',
        r'\btq\b': 'taluka',
        r'\br\.\b': 'rue',
    }
    for pattern, replacement in replacements.items():
        s = re.sub(pattern, replacement, s)
    
    # Remove punctuation
    s = re.sub(r'[^\w\s]', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s


def preprocess_df(df):
    """Apply all preprocessing to a dataframe."""
    df = df.copy()
    df['business_name'] = df['business_name'].fillna('')
    df['business_address'] = df['business_address'].fillna('')
    df['name_norm'] = df['business_name'].apply(normalize_name)
    df['addr_norm'] = df['business_address'].apply(normalize_address)
    # Combined text for TF-IDF blocking
    df['combined_text'] = df['name_norm'] + ' ' + df['addr_norm']
    return df

print("Preprocessing functions defined ✓")

In [ ]:
# Apply preprocessing to all sources
print("Preprocessing training data...")
train_s1 = preprocess_df(train_s1)
train_s2 = preprocess_df(train_s2)
train_s3 = preprocess_df(train_s3)

print("Preprocessing test data...")
test_s1 = preprocess_df(test_s1)
test_s2 = preprocess_df(test_s2)
test_s3 = preprocess_df(test_s3)

print("\nSample preprocessed data:")
display(train_s1[['entity_id', 'business_name', 'name_norm', 'business_address', 'addr_norm', 'country']].head(3))

## 4. Validation Split

We hold out a portion of the training data to evaluate our pipeline using the same F₀.₅ metric as the leaderboard.

In [ ]:
def compute_f05_score(predictions, ground_truth):
    """
    Compute macro-averaged F_0.5 score.
    
    predictions: dict {s1_id: set of matched ids}
    ground_truth: dict {s1_id: set of matched ids}
    """
    scores = []
    for s1_id in ground_truth:
        true = ground_truth[s1_id]
        pred = predictions.get(s1_id, set())
        
        if len(true) == 0 and len(pred) == 0:
            scores.append(1.0)  # Correct singleton
        elif len(true) == 0 and len(pred) > 0:
            scores.append(0.0)  # False merge on singleton
        elif len(pred) == 0 and len(true) > 0:
            scores.append(0.0)  # Missed all matches
        else:
            tp = len(pred & true)
            precision = tp / len(pred) if len(pred) > 0 else 0
            recall = tp / len(true) if len(true) > 0 else 0
            if precision + recall == 0:
                scores.append(0.0)
            else:
                f05 = (1.25 * precision * recall) / (0.25 * precision + recall)
                scores.append(f05)
    
    return np.mean(scores)


# Parse ground truth into dict
def parse_ground_truth(gt_df):
    """Convert ground truth df to dict {s1_id: set of matched ids}."""
    gt_dict = {}
    for _, row in gt_df.iterrows():
        s1_id = row['source1_entity_id']
        if pd.isna(row['matched_entity_ids']):
            gt_dict[s1_id] = set()
        else:
            gt_dict[s1_id] = set(row['matched_entity_ids'].split(','))
    return gt_dict


# Create validation split (use 5% of S1 entities for validation due to scale)
VAL_FRAC = 0.05
val_mask = np.random.rand(len(train_gt)) < VAL_FRAC
val_gt = train_gt[val_mask].reset_index(drop=True)
train_gt_subset = train_gt[~val_mask].reset_index(drop=True)

val_gt_dict = parse_ground_truth(val_gt)
val_s1_ids = set(val_gt['source1_entity_id'])

print(f"Validation S1 entities: {len(val_gt):,}")
print(f"Training S1 entities (remaining): {len(train_gt_subset):,}")
print(f"Validation singletons: {sum(1 for v in val_gt_dict.values() if len(v)==0):,}")
print(f"\nF₀.₅ scoring function defined ✓")

## 5. Blocking / Candidate Generation

At this scale (~2M × 10M), brute-force all-pairs comparison is infeasible. We use **TF-IDF cosine similarity** with **country-based partitioning** for efficient blocking:

1. **Partition by country** — entities can only match within the same country
2. **TF-IDF vectorize** the combined (name + address) text
3. **Top-K cosine similarity** — for each S1 entity, find the top-K most similar S2/S3 candidates
4. This produces the candidate set that feeds into the matching model

This is the most compute-intensive step. We process in batches to manage memory.

In [ ]:
def build_candidates_for_country(s1_df, s2_df, s3_df, country, top_k=20, batch_size=5000):
    """
    Build candidate pairs for a single country using TF-IDF + cosine similarity.
    
    Returns: dict {s1_id: list of candidate ids from S2/S3}
    """
    # Filter by country
    s1_c = s1_df[s1_df['country'] == country].reset_index(drop=True)
    s2_c = s2_df[s2_df['country'] == country].reset_index(drop=True)
    s3_c = s3_df[s3_df['country'] == country].reset_index(drop=True)
    
    if len(s1_c) == 0:
        return {}
    
    # Combine S2 + S3 as the candidate pool
    candidates = pd.concat([s2_c, s3_c], ignore_index=True)
    
    if len(candidates) == 0:
        return {eid: [] for eid in s1_c['entity_id']}
    
    print(f"  Country '{country}': S1={len(s1_c):,}, Candidates={len(candidates):,}")
    
    # Fit TF-IDF on combined corpus
    all_texts = pd.concat([s1_c['combined_text'], candidates['combined_text']], ignore_index=True)
    
    tfidf = TfidfVectorizer(
        analyzer='char_wb',
        ngram_range=(3, 5),
        max_features=100000,
        sublinear_tf=True,
        dtype=np.float32
    )
    tfidf.fit(all_texts)
    
    # Transform candidates once
    cand_vectors = tfidf.transform(candidates['combined_text'])
    cand_ids = candidates['entity_id'].values
    
    # Process S1 in batches
    result = {}
    n_batches = (len(s1_c) + batch_size - 1) // batch_size
    
    for batch_idx in tqdm(range(n_batches), desc=f"  Blocking ({country})", leave=False):
        start = batch_idx * batch_size
        end = min(start + batch_size, len(s1_c))
        batch_s1 = s1_c.iloc[start:end]
        
        s1_vectors = tfidf.transform(batch_s1['combined_text'])
        
        # Compute cosine similarity
        sim_matrix = cosine_similarity(s1_vectors, cand_vectors)
        
        # Get top-K candidates per S1 entity
        for i in range(len(batch_s1)):
            s1_id = batch_s1.iloc[i]['entity_id']
            sims = sim_matrix[i]
            
            # Get top-K indices
            if len(sims) <= top_k:
                top_indices = np.argsort(sims)[::-1]
            else:
                top_indices = np.argpartition(sims, -top_k)[-top_k:]
                top_indices = top_indices[np.argsort(sims[top_indices])[::-1]]
            
            # Filter out very low similarity (noise)
            threshold = 0.05
            top_cands = [(cand_ids[idx], sims[idx]) for idx in top_indices if sims[idx] > threshold]
            result[s1_id] = [cid for cid, _ in top_cands]
    
    # Clean up
    del cand_vectors, tfidf
    
    return result

print("Blocking function defined ✓")

### 5.1 Validation Blocking

Run blocking on the validation subset first to tune parameters and evaluate blocking recall (the upper bound of our system's recall).

In [ ]:
# Get validation S1 records
val_s1_df = train_s1[train_s1['entity_id'].isin(val_s1_ids)].reset_index(drop=True)
val_countries = val_s1_df['country'].unique()

print(f"Running blocking on {len(val_s1_df):,} validation S1 entities...")
print(f"Countries: {val_countries}")

val_candidates = {}
for country in val_countries:
    country_cands = build_candidates_for_country(
        val_s1_df, train_s2, train_s3, country, top_k=20, batch_size=10000
    )
    val_candidates.update(country_cands)

# For S1 entities not in any country partition (shouldn't happen), add empty
for s1_id in val_s1_ids:
    if s1_id not in val_candidates:
        val_candidates[s1_id] = []

# Evaluate blocking recall
total_true_matches = 0
found_by_blocking = 0
for s1_id, true_matches in val_gt_dict.items():
    if len(true_matches) == 0:
        continue
    total_true_matches += len(true_matches)
    cand_set = set(val_candidates.get(s1_id, []))
    found_by_blocking += len(true_matches & cand_set)

blocking_recall = found_by_blocking / total_true_matches if total_true_matches > 0 else 0
avg_candidates = np.mean([len(v) for v in val_candidates.values()])

print(f"\n=== Blocking Evaluation ===")
print(f"Blocking Recall: {blocking_recall:.4f} (upper bound for final recall)")
print(f"Avg candidates per S1 entity: {avg_candidates:.1f}")
print(f"Total candidate pairs: {sum(len(v) for v in val_candidates.values()):,}")

## 6. Feature Engineering

For each candidate pair (S1, S2/S3), we compute multiple similarity features:
- **Name similarity:** Token Sort Ratio, Token Set Ratio, Partial Ratio, Levenshtein ratio
- **Address similarity:** Token Sort Ratio, Token Set Ratio, Partial Ratio
- **Combined similarity:** TF-IDF cosine (already computed during blocking)

In [ ]:
def compute_pair_features(s1_row, cand_row):
    """Compute similarity features for a single (S1, candidate) pair."""
    name1 = s1_row['name_norm']
    name2 = cand_row['name_norm']
    addr1 = s1_row['addr_norm']
    addr2 = cand_row['addr_norm']
    
    features = {}
    
    # Name features
    features['name_token_sort'] = fuzz.token_sort_ratio(name1, name2) / 100.0
    features['name_token_set'] = fuzz.token_set_ratio(name1, name2) / 100.0
    features['name_partial'] = fuzz.partial_ratio(name1, name2) / 100.0
    features['name_ratio'] = fuzz.ratio(name1, name2) / 100.0
    
    # Address features
    if addr1 and addr2:
        features['addr_token_sort'] = fuzz.token_sort_ratio(addr1, addr2) / 100.0
        features['addr_token_set'] = fuzz.token_set_ratio(addr1, addr2) / 100.0
        features['addr_partial'] = fuzz.partial_ratio(addr1, addr2) / 100.0
    else:
        features['addr_token_sort'] = 0.0
        features['addr_token_set'] = 0.0
        features['addr_partial'] = 0.0
    
    # Combined weighted score (simple but effective)
    features['combined_score'] = (
        0.40 * features['name_token_sort'] +
        0.15 * features['name_token_set'] +
        0.10 * features['name_partial'] +
        0.20 * features['addr_token_sort'] +
        0.10 * features['addr_token_set'] +
        0.05 * features['addr_partial']
    )
    
    return features

print("Feature engineering functions defined ✓")

## 7. Matching

We use a **threshold-based approach** on the combined similarity score. Since F₀.₅ is precision-heavy, we tune the threshold to favor precision over recall.

The strategy:
- Compute features for each candidate pair
- Apply threshold on `combined_score` to make match decisions
- Tune threshold on validation set to maximize F₀.₅

In [ ]:
def score_candidates(s1_df, s23_df, candidates_dict, threshold=0.5):
    """
    Score candidate pairs and return matches above threshold.
    
    Args:
        s1_df: Source 1 dataframe (preprocessed)
        s23_df: Combined Source 2+3 dataframe (preprocessed)
        candidates_dict: {s1_id: [candidate_ids]}
        threshold: minimum combined_score to be considered a match
    
    Returns:
        matches: dict {s1_id: set of matched ids}
    """
    # Build lookup dicts for fast access
    s1_lookup = s1_df.set_index('entity_id')
    s23_lookup = s23_df.set_index('entity_id')
    
    matches = {}
    
    all_s1_ids = list(candidates_dict.keys())
    
    for s1_id in tqdm(all_s1_ids, desc="Scoring pairs", leave=False):
        cand_ids = candidates_dict[s1_id]
        matched = set()
        
        if len(cand_ids) == 0:
            matches[s1_id] = matched
            continue
        
        s1_row = s1_lookup.loc[s1_id]
        
        for cand_id in cand_ids:
            if cand_id not in s23_lookup.index:
                continue
            cand_row = s23_lookup.loc[cand_id]
            
            features = compute_pair_features(s1_row, cand_row)
            
            if features['combined_score'] >= threshold:
                matched.add(cand_id)
        
        matches[s1_id] = matched
    
    return matches

print("Matching functions defined ✓")

### 7.1 Threshold Tuning on Validation Set

We try multiple thresholds and pick the one that maximizes F₀.₅ on the validation set.

In [ ]:
# Combine S2 + S3 for lookup
train_s23 = pd.concat([train_s2, train_s3], ignore_index=True)

# Score validation candidates at different thresholds
print("Scoring validation candidates...")
thresholds = [0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]
best_f05 = 0
best_threshold = 0.5

# First, compute all features for validation pairs
print("Computing features for all validation candidate pairs...")
val_pair_scores = {}

s1_lookup = val_s1_df.set_index('entity_id')
s23_lookup = train_s23.set_index('entity_id')

for s1_id in tqdm(val_candidates, desc="Feature computation"):
    cand_ids = val_candidates[s1_id]
    if len(cand_ids) == 0:
        val_pair_scores[s1_id] = {}
        continue
    
    s1_row = s1_lookup.loc[s1_id]
    scores = {}
    for cand_id in cand_ids:
        if cand_id not in s23_lookup.index:
            continue
        cand_row = s23_lookup.loc[cand_id]
        features = compute_pair_features(s1_row, cand_row)
        scores[cand_id] = features['combined_score']
    val_pair_scores[s1_id] = scores

print("\nThreshold tuning:")
print(f"{'Threshold':>10} | {'Precision':>10} | {'Recall':>10} | {'F_0.5':>10}")
print("-" * 50)

for thresh in thresholds:
    predictions = {}
    for s1_id in val_pair_scores:
        matched = set()
        for cand_id, score in val_pair_scores[s1_id].items():
            if score >= thresh:
                matched.add(cand_id)
        predictions[s1_id] = matched
    
    f05 = compute_f05_score(predictions, val_gt_dict)
    
    # Also compute precision and recall for insight
    total_pred = sum(len(v) for v in predictions.values())
    total_true = sum(len(v) for v in val_gt_dict.values())
    tp_total = sum(len(predictions.get(k, set()) & v) for k, v in val_gt_dict.items())
    precision = tp_total / total_pred if total_pred > 0 else 0
    recall = tp_total / total_true if total_true > 0 else 0
    
    print(f"{thresh:>10.2f} | {precision:>10.4f} | {recall:>10.4f} | {f05:>10.4f}")
    
    if f05 > best_f05:
        best_f05 = f05
        best_threshold = thresh

print(f"\n✓ Best threshold: {best_threshold} with F₀.₅ = {best_f05:.4f}")

## 8. Test Set Inference

Now we apply the full pipeline to the test data:
1. Block/generate candidates per country
2. Score each candidate pair
3. Apply the tuned threshold
4. Generate both output files

In [ ]:
# Get all countries in test set
test_countries = sorted(set(test_s1['country'].unique()) | 
                        set(test_s2['country'].unique()) | 
                        set(test_s3['country'].unique()))
print(f"Test countries: {test_countries}")
print(f"Note: 'France' is new — not in training data\n")

# Run blocking for each country
print("=== Candidate Generation (Test Set) ===")
test_candidates = {}

for country in test_countries:
    country_cands = build_candidates_for_country(
        test_s1, test_s2, test_s3, country, top_k=20, batch_size=10000
    )
    test_candidates.update(country_cands)

# Ensure every test S1 entity has an entry
for eid in test_s1['entity_id']:
    if eid not in test_candidates:
        test_candidates[eid] = []

print(f"\nTotal test S1 entities with candidates: {len(test_candidates):,}")
print(f"Avg candidates per entity: {np.mean([len(v) for v in test_candidates.values()]):.1f}")

In [ ]:
# Score and match test candidates
print("=== Matching (Test Set) ===")
test_s23 = pd.concat([test_s2, test_s3], ignore_index=True)
test_matches = score_candidates(test_s1, test_s23, test_candidates, threshold=best_threshold)

# Stats
n_matched = sum(1 for v in test_matches.values() if len(v) > 0)
n_singleton = sum(1 for v in test_matches.values() if len(v) == 0)
total_match_ids = sum(len(v) for v in test_matches.values())

print(f"\n=== Test Results ===")
print(f"S1 entities with matches: {n_matched:,}")
print(f"S1 singletons (no match): {n_singleton:,}")
print(f"Total matched IDs: {total_match_ids:,}")
print(f"Avg matches per entity: {total_match_ids/len(test_matches):.2f}")

## 9. Generate Output Files

We produce both required files:
1. **`matching_results.tsv`** — final matches (scored on leaderboard)
2. **`candidate_pairs.tsv`** — blocking candidate set (for pipeline verification)

In [ ]:
# --- matching_results.tsv ---
matching_rows = []
for s1_id in test_s1['entity_id']:
    matched = test_matches.get(s1_id, set())
    matched_str = ','.join(sorted(matched)) if matched else ''
    matching_rows.append({'source1_entity_id': s1_id, 'matched_entity_ids': matched_str})

matching_df = pd.DataFrame(matching_rows)
matching_df.to_csv(os.path.join(OUTPUT_DIR, "matching_results.tsv"), sep="\t", index=False)
print(f"✓ matching_results.tsv written ({len(matching_df):,} rows)")

# --- candidate_pairs.tsv ---
candidate_rows = []
for s1_id in test_s1['entity_id']:
    cands = test_candidates.get(s1_id, [])
    cands_str = ','.join(sorted(cands)) if cands else ''
    candidate_rows.append({'source1_entity_id': s1_id, 'candidate_entity_ids': cands_str})

candidate_df = pd.DataFrame(candidate_rows)
candidate_df.to_csv(os.path.join(OUTPUT_DIR, "candidate_pairs.tsv"), sep="\t", index=False)
print(f"✓ candidate_pairs.tsv written ({len(candidate_df):,} rows)")

# Verify matches are subset of candidates
print("\n=== Sanity Check ===")
violations = 0
for s1_id in test_matches:
    matched = test_matches[s1_id]
    cands = set(test_candidates.get(s1_id, []))
    if not matched.issubset(cands):
        violations += 1
print(f"Match-not-in-candidates violations: {violations}")

## 10. Validate Submission

Run the provided validation script to ensure our output files are correctly formatted.

In [ ]:
import subprocess
result = subprocess.run(
    ["python3", "utils/validate_submission.py",
     "--matching", "../output/matching_results.tsv",
     "--candidate", "../output/candidate_pairs.tsv",
     "--test-dir", "dataset/test"],
    capture_output=True, text=True,
    cwd="student_resource"
)
print("STDOUT:", result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)
print(f"Exit code: {result.returncode}")

## 11. Output Preview

In [ ]:
# Preview matching results
print("=== matching_results.tsv (first 10 rows) ===")
preview = pd.read_csv(os.path.join(OUTPUT_DIR, "matching_results.tsv"), sep="\t", nrows=10)
display(preview)

print("\n=== candidate_pairs.tsv (first 10 rows) ===")
preview_cand = pd.read_csv(os.path.join(OUTPUT_DIR, "candidate_pairs.tsv"), sep="\t", nrows=10)
display(preview_cand)

## 📊 Summary

### Pipeline Architecture
1. **Preprocessing:** Normalized business names (legal suffix standardization) and addresses (abbreviation expansion)
2. **Blocking:** Country-partitioned TF-IDF (char 3-5 grams) + cosine similarity → top-20 candidates per S1 entity
3. **Feature Engineering:** 7 string similarity features (token sort, token set, partial ratio for name & address)
4. **Matching:** Weighted combined score with threshold tuned on validation set for F₀.₅
5. **Output:** Both `matching_results.tsv` and `candidate_pairs.tsv`

### Key Design Decisions
- **Character n-gram TF-IDF** handles transliterations, typos, and abbreviations better than word-level
- **Country partitioning** reduces search space by ~50-60% since cross-country matches are impossible
- **Precision-heavy threshold** aligned with F₀.₅ metric — we'd rather miss a match than create a false merge
- **No external APIs or data** — pure ML approach using only provided training data

### Possible Improvements
- Use a learned ranking model (LightGBM/XGBoost) instead of weighted threshold
- Add more blocking strategies (exact name tokens, phonetic codes)
- Use sentence transformers for semantic similarity on names
- Ensemble multiple blocking approaches to improve recall ceiling
- Fine-tune separately for each country's patterns